In [1]:
import sqlite3
conn = sqlite3.connect(":memory:")   
cur = conn.cursor()

In [2]:
# --- 셀 A: FBref 캐시에서 데이터를 꺼내 어떻게 생겼는지 확인 ---
# 목적: DB에 뭘 넣을지 정하기 전에, 재료(컬럼들)를 눈으로 본다.
# 이 셀은 아무것도 만들지/바꾸지 않음. 읽고 출력만 함.

import sys; sys.path.append("..")   # 노트북에서 프로젝트 폴더를 보게 하는 늘 그 줄
import soccerdata as sd             # FBref 캐시를 읽을 줄 아는 라이브러리 (Day 1의 그 배달부)

# 캐시에서 EPL 24-25 시즌 선수 스탯을 표(DataFrame) 형태로 로드
fbref = sd.FBref(leagues="ENG-Premier League", seasons="2425")
df = fbref.read_player_season_stats()

# 표의 행 이름표(인덱스)를 일반 열로 바꿔서 다루기 쉽게 만듦
df = df.reset_index()

# 열 이름 전부 출력 + 위에서 3줄 미리보기
print(df.columns.tolist())
print(df.head(3))

[07/16/26 22:34:28] INFO     No custom team name replacements found. You can configure these in       ]8;id=9719196;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=9719197;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_config.py#92\92]8;;\
                             /Users/jade/soccerdata/config/teamname_replacements.json.                             

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=9719203;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=9719204;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_config.py#190\190]8;;\
                             /Users/jade/soccerdata/config/league_dict.json.                                       

                    INFO     Saving cached data to /Users/jade/soccerdata/data/FBref                 ]8;id=9719211;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=9719212;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_common.py#250\250]8;;\

[('league', ''), ('season', ''), ('team', ''), ('player', ''), ('nation', ''), ('pos', ''), ('age', ''), ('born', ''), ('Playing Time', 'MP'), ('Playing Time', 'Starts'), ('Playing Time', 'Min'), ('Playing Time', '90s'), ('Performance', 'Gls'), ('Performance', 'Ast'), ('Performance', 'G+A'), ('Performance', 'G-PK'), ('Performance', 'PK'), ('Performance', 'PKatt'), ('Performance', 'CrdY'), ('Performance', 'CrdR'), ('Per 90 Minutes', 'Gls'), ('Per 90 Minutes', 'Ast'), ('Per 90 Minutes', 'G+A'), ('Per 90 Minutes', 'G-PK'), ('Per 90 Minutes', 'G+A-PK')]
               league season     team       player nation    pos age  born  \
                                                                             
0  ENG-Premier League   2425  Arsenal    Ben White    ENG     DF  26  1997   
1  ENG-Premier League   2425  Arsenal  Bukayo Saka    ENG  FW,MF  22  2001   
2  ENG-Premier League   2425  Arsenal   David Raya    ESP     GK  28  1995   

  Playing Time         ... Performance               

In [3]:
# --- Cell B: move the 9 chosen columns into data/stats.db ---
# What this cell does, in order:
#   1) pick 9 items from the warehouse (df)
#   2) give them short names
#   3) pour them into the SQL container (stats.db) — the actual "moving"
#   4) count what arrived

import sqlite3

# 1) pick — the two-part names come straight from Cell A's output
players = df[[("player", ""),
              ("team", ""),
              ("pos", ""),
              ("age", ""),
              ("Playing Time", "Min"),
              ("Performance", "Gls"),
              ("Performance", "Ast"),
              ("Per 90 Minutes", "G+A"),
              ("Performance", "PK")]].copy()

# 2) rename — short single words that SQL can use easily
players.columns = ["name", "team", "position", "age", "minutes",
                   "goals", "assists", "ga_per90", "pk"]

# 3) pour into the SQL container — this line CREATES the file data/stats.db
conn = sqlite3.connect("../data/stats.db")
players.to_sql("players", conn, if_exists="replace", index=False)

# 4) count — expect 500-ish (every EPL player from that season)
print(conn.execute("SELECT COUNT(*) FROM players").fetchone())

(574,)


In [4]:
# --- Cell C: the first real scouting query, raw SQL ---
# Question: "young forwards, best goal contribution per 90 minutes"
# = the opening query of the Son-replacement search.
# Read the SQL like English: SELECT (show these columns) FROM players
# WHERE (only rows matching these conditions) ORDER BY (sort) LIMIT (top 10).

rows = conn.execute("""
    SELECT name, team, position, age, minutes, goals, assists, ga_per90
    FROM players
    WHERE position LIKE '%FW%'      -- position contains 'FW' (Saka is 'FW,MF')
      AND age <= 25                 -- young enough to build around
      AND minutes >= 900            -- at least 10 full games (no small-sample flukes)
    ORDER BY ga_per90 DESC          -- best goal contribution first
    LIMIT 10
""").fetchall()

for r in rows:
    print(r)
    

('Alexander Isak', 'Newcastle', 'FW', 24, 2756, 23, 6, 0.95)
('Rodrigo Muniz', 'Fulham', 'FW', 23, 964, 8, 1, 0.84)
('Bukayo Saka', 'Arsenal', 'FW,MF', 22, 1729, 6, 10, 0.83)
('Erling Haaland', 'Manchester City', 'FW', 24, 2736, 22, 3, 0.82)
('João Pedro', 'Brighton', 'FW,MF', 22, 1948, 10, 6, 0.74)
('Bryan Mbeumo', 'Brentford', 'MF,FW', 24, 3414, 20, 7, 0.71)
('Amad Diallo', 'Manchester Utd', 'MF,FW', 22, 1901, 8, 6, 0.66)
('Jørgen Strand Larsen', 'Wolves', 'FW', 24, 2587, 14, 4, 0.63)
('Nicolas Jackson', 'Chelsea', 'FW', 23, 2220, 10, 5, 0.61)
('Anthony Elanga', 'Nottingham', 'MF,FW', 22, 2501, 6, 11, 0.61)


In [5]:
# --- Cell D: pull two more boxes from the warehouse: defense & passing ---
# Same warehouse (FBref cache), different stat tables.
# LOOK first, pick columns after — same routine as Cell A.

df_def = fbref.read_player_season_stats(stat_type="defense").reset_index()
df_pas = fbref.read_player_season_stats(stat_type="passing").reset_index()

print("DEFENSE columns:", df_def.columns.tolist())
print("PASSING columns:", df_pas.columns.tolist())

TypeError: Invalid argument: stat_type should be in ['standard', 'keeper', 'shooting', 'playing_time', 'misc']

In [3]:
# --- Cell D (v3): retry with a visible browser window ---
# headless=False = the automated Chrome actually shows on screen.
# Cloudflare trusts a visible browser more than an invisible one.
# A Chrome window will pop up and drive itself — do not close it.

import soccerdata as sd

fbref_v = sd.FBref(leagues="ENG-Premier League", seasons="2425", headless=False)

df_misc = fbref_v.read_player_season_stats(stat_type="misc").reset_index()
print("MISC columns:", df_misc.columns.tolist())

[07/16/26 23:09:10] INFO     No custom team name replacements found. You can configure these in       ]8;id=15647970;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=15647971;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_config.py#92\92]8;;\
                             /Users/jade/soccerdata/config/teamname_replacements.json.                             

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=15647977;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=15647978;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_config.py#190\190]8;;\
                             /Users/jade/soccerdata/config/league_dict.json.                                       

                    INFO     Saving cached data to /Users/jade/soccerdata/data/FBref                 ]8;id=15647985;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=15647986;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_common.py#250\250]8;;\

MISC columns: [('league', ''), ('season', ''), ('team', ''), ('player', ''), ('nation', ''), ('pos', ''), ('age', ''), ('born', ''), ('90s', ''), ('Performance', 'CrdY'), ('Performance', 'CrdR'), ('Performance', '2CrdY'), ('Performance', 'Fls'), ('Performance', 'Fld'), ('Performance', 'Off'), ('Performance', 'Crs'), ('Performance', 'Int'), ('Performance', 'TklW'), ('Performance', 'PKwon'), ('Performance', 'PKcon'), ('Performance', 'OG')]


In [8]:
# --- Cell G: wrap this morning's raw SQL into a reusable function ---
# What it does: scouting criteria in -> ranked player list out.
# Today it ranks by attacking output (goals + assists per 90).
# When the misc table arrives, DF/MF ranking modes get added HERE —
# the function is built so that is a small change, not a rewrite.

import sqlite3

def find_players(position=None, max_age=None, min_minutes=900, limit=10):
    """Search players by scouting criteria, best candidates first.

    min_minutes=900 (10 full games) keeps small-sample flukes out.
    """
    conn = sqlite3.connect("../data/stats.db")
    conn.row_factory = sqlite3.Row   # lets us read columns by name

    query = """
        SELECT name, team, position, age, minutes, goals, assists,
               ROUND((goals + assists) * 90.0 / minutes, 2) AS score
        FROM players
        WHERE minutes >= ?
    """
    params = [min_minutes]

    # optional filters, added only if the caller asks
    if position:
        query += " AND position LIKE ?"
        params.append(f"%{position}%")
    if max_age:
        query += " AND age <= ?"
        params.append(max_age)

    query += " ORDER BY score DESC LIMIT ?"
    params.append(limit)

    rows = conn.execute(query, params).fetchall()
    conn.close()
    return [dict(r) for r in rows]


# quick test — should match this morning's top-10 list
for p in find_players(position="FW", max_age=25):
    print(p["name"], p["team"], p["age"], p["score"])

Alexander Isak Newcastle 24 0.95
Rodrigo Muniz Fulham 23 0.84
Bukayo Saka Arsenal 22 0.83
Erling Haaland Manchester City 24 0.82
João Pedro Brighton 22 0.74
Bryan Mbeumo Brentford 24 0.71
Amad Diallo Manchester Utd 22 0.66
Jørgen Strand Larsen Wolves 24 0.63
Nicolas Jackson Chelsea 23 0.61
Anthony Elanga Nottingham 22 0.61


In [1]:
# --- compatibility check: same shapes as the old loader, new engine inside ---
import sys; sys.path.append("..")
from data.loader import get_player_stats, find_players

print(get_player_stats("Saka"))        # 1 record, same envelope shape
print(get_player_stats("Rashford"))    # 2 records — transfer split preserved!
print(get_player_stats("Nobody"))      # {"error": ...}
print(find_players(position="FW", max_age=25)[0])   # Isak should top the list

{'player': 'Bukayo Saka', 'season': '2425', 'competition': 'ENG-Premier League', 'records': [{'team': 'Arsenal', 'stats': {'position': 'FW,MF', 'age': 22, 'minutes': 1729, 'goals': 6, 'assists': 10, 'ga_per90': 0.83, 'pk': 1}}]}
{'player': 'Marcus Rashford', 'season': '2425', 'competition': 'ENG-Premier League', 'records': [{'team': 'Aston Villa', 'stats': {'position': 'FW,MF', 'age': 26, 'minutes': 444, 'goals': 2, 'assists': 2, 'ga_per90': 0.81, 'pk': 1}}, {'team': 'Manchester Utd', 'stats': {'position': 'MF', 'age': 26, 'minutes': 978, 'goals': 4, 'assists': 1, 'ga_per90': 0.46, 'pk': 0}}]}
{'error': "Player 'Nobody' not found"}
{'name': 'Alexander Isak', 'team': 'Newcastle', 'position': 'FW', 'age': 24, 'minutes': 2756, 'goals': 23, 'assists': 6, 'score': 0.95}


In [2]:
# --- final gate: the upper layers must not notice the engine swap ---
# Agent, both judges, pipeline: untouched today. If this passes,
# the abstraction layer proved itself.

from orchestrator.pipeline import scout_pipeline
print(scout_pipeline("How many goals did Saka score this season?"))


  [tool call] get_player_stats({'name': 'Saka', 'season': '2425'})
{'answer': 'Bukayo Saka scored 6 goals for Arsenal in the 2024-25 Premier League season.', 'verdict': 'pass', 'details': [{'verdict': 'pass', 'claimed': [6.0], 'unverified': []}], 'reasoning_check': {'verdict': 'pass', 'reasoning': "The agent's answer accurately states information directly present in the provided data without making any unsupported claims or drawing conclusions that do not follow.", 'violations': []}}


In [4]:
# --- Cell E: pour the defensive picks from misc into stats.db ---
# name+team = the JOIN key (transfers stay split, Day 2 rule).
# "90s" comes along for per-90 math.

import sqlite3

defense = df_misc[[("player", ""),
                   ("team", ""),
                   ("90s", ""),
                   ("Performance", "TklW"),
                   ("Performance", "Int"),
                   ("Performance", "Fls"),
                   ("Performance", "CrdY")]].copy()
defense.columns = ["name", "team", "nineties",
                   "tackles_won", "interceptions", "fouls", "yellow_cards"]

conn = sqlite3.connect("../data/stats.db")
defense.to_sql("defense", conn, if_exists="replace", index=False)
print(conn.execute("SELECT COUNT(*) FROM defense").fetchone())   # expect ~574

(574,)


In [5]:
# --- Cell F: first defender search — young DFs by defensive actions per 90 ---
rows = conn.execute("""
    SELECT p.name, p.team, p.age, p.minutes,
           d.tackles_won, d.interceptions,
           ROUND((d.tackles_won + d.interceptions) / d.nineties, 2) AS def_score
    FROM players p
    JOIN defense d ON p.name = d.name AND p.team = d.team
    WHERE p.position LIKE '%DF%'
      AND p.age <= 25
      AND p.minutes >= 900
    ORDER BY def_score DESC
    LIMIT 10
""").fetchall()

for r in rows:
    print(r)

('Mats Wieffer', 'Brighton', 24, 1015, 34, 16, 4.42)
('Luke Thomas', 'Leicester City', 23, 1110, 25, 20, 3.66)
('Moisés Caicedo', 'Chelsea', 22, 3351, 74, 47, 3.25)
('Victor Bernth Kristiansen', 'Leicester City', 21, 2482, 51, 37, 3.19)
('Destiny Udogie', 'Tottenham', 21, 1924, 45, 21, 3.08)
('Trent Alexander-Arnold', 'Liverpool', 25, 2365, 49, 31, 3.04)
('Neco Williams', 'Nottingham', 23, 2594, 57, 30, 3.02)
('Trevoh Chalobah', 'Crystal Palace', 25, 1060, 15, 20, 2.97)
('James Garner', 'Everton', 23, 1594, 32, 20, 2.94)
('Ian Maatsen', 'Aston Villa', 22, 1139, 18, 18, 2.83)


In [ ]:
import sys; sys.path.append("..")
from data.loader import find_players
print(find_players(position="DF", max_age=25)[0])   # Wieffer 기대
print(find_players(position="FW", max_age=25)[0])   # Isak 기대